#Cmd to push into GIT

In [43]:
import os
from google.colab import userdata

# Set up your git config details
os.environ["GITHUB_USERNAME"] = "aayushks2003"
os.environ["GITHUB_EMAIL"] = "06719011621.aiml@gmail.com"
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

# Apply the global config to git
!git config --global user.name $GITHUB_USERNAME
!git config --global user.email $GITHUB_EMAIL

print("Git profile configuration complete and secure!")

Git profile configuration complete and secure!


In [44]:
# 1. Initialize a local git repository
!git init

# Copy into git file
!cp "/content/drive/MyDrive/transcript/Rag.ipynb" ./Rag.ipynb
!cp "/content/drive/MyDrive/transcript/transcript.txt" ./transcript.txt

# add to the
!git add Rag.ipynb transcript.txt

# Check the status to verify they turn GREEN (staged)
!git status


Reinitialized existing Git repository in /content/.git/
On branch main
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   Rag.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	drive/
	sample_data/



In [45]:
import os

# Re-commit the targeted local workspace files
!git commit -m "Secure deployment of RAG pipeline notebook and transcript"

# Re-verify branch targeting
!git branch -M main

# Pull down the remote history and blend it with our local files using 'allow-unrelated-histories'
!git pull origin main --allow-unrelated-histories --no-edit

# Push your synchronized repository up to GitHub!
!git push -f origin main

[main a672e5f] Secure deployment of RAG pipeline notebook and transcript
 1 file changed, 1 insertion(+), 1 deletion(-)
 rewrite Rag.ipynb (93%)
From https://github.com/aayushks2003/RAG-on-youtube-transcript
 * branch            main       -> FETCH_HEAD
Already up to date.
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 1.38 KiB | 1.38 MiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/aayushks2003/RAG-on-youtube-transcript.git
   cc22242..a672e5f  main -> main


#Install Dependencies

In [ ]:
!pip install -q langchain langchain-core langchain-community chromadb streamlit huggingface-hub transformers langchain-huggingface langchain-openai faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
os.environ["OPENAI_API_KEY"]=userdata.get('OPENAI_API_KEY')

#Indexing

###Ingestion

In [25]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('/content/drive/MyDrive/transcript/transcript.txt', encoding='utf-8')
document = loader.load()
type(document)

list

###Text Splitting


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=60)
chunk = splitter.split_documents(document)
chunk[99]

Document(metadata={'source': '/content/drive/MyDrive/trnscrpt/trnscript.txt'}, page_content="39:2739 minutes, 27 secondslet's also just remove this bit for now so we can just see what the strictured llm outputs directly and let's\n39:3639 minutes, 36 secondssee okay so now you can see that we actually have that paragraph object right the one we defined up here which is kind of cool and then in there we\n39:4539 minutes, 45 secondshave the original paragraph right so this is where this is coming from I definitely remember writing something")

###Embedding

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

###Vector Storage

In [ ]:
from langchain_community.vectorstores import FAISS

vectorStore = FAISS.from_documents(chunk, embeddings)

###Retriver

In [ ]:
retriever = vectorStore.as_retriever(search_type="similarity", search_kwarg={"k":4})

context2 = retriever.invoke("What is this about")

context2[0].page_content

"21:2721 minutes, 27 secondsgoing to be using line chain to generate various sces that we might um find helpful as we're you know we have this\n21:3421 minutes, 34 secondsarticle draft and we're editing it and just kind of like finalizing it so what are those going to be you can see them here we have the title for the article\n21:4321 minutes, 43 secondsthe description and SEO friendly description specifically third one we're going to be getting the LM to Providers"

#Augumentation/Generation

###Prompts

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
    You are an helpful AI assistant.
    Use the {context} to answer the {question}. Give the answer in understandable english.
    If the context is insufficient, just say that you don't know.
    """,
    input_variables=["context", "question"]
)

###LLM Model

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model = "gpt-5.4-mini-2026-03-17",
    temperature = 0.3
)


###String Outparser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

#Chains / Runnables output

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough


generator_chain = prompt | model | output_parser

retrieval_chain= RunnableParallel({
    "context":retriever,
    "question":RunnablePassthrough()
    })

main_chain = retrieval_chain | generator_chain

result=main_chain.invoke("What is this about")

result

'This appears to be a transcript from a course or tutorial about **LangChain** and related tools for building AI applications.\n\nFrom the snippets, it looks like the content covers:\n- **Getting started with LangChain**\n- Building simple **LLM-powered assistants**\n- Generating things like **text, images, and structured outputs**\n- Using LangChain to help with **editing an article draft**, such as creating titles, descriptions, and SEO-friendly descriptions\n- Later chapters move into **LangSmith**, which is used for **AI observability** and tracking/debugging applications\n\nSo, in simple English: **it’s about learning how to use LangChain to build AI apps and then using LangSmith to observe and improve them.**'

Input

In [ ]:
import streamlit as st


